In [1]:
from pathlib import Path  # Dùng để tìm thư mục dự án mà không ghi đường dẫn cố định.
import sys  # Dùng để Python tìm thấy các module trong thư mục src/.

project_root = Path.cwd()  # Lấy thư mục hiện tại của kernel notebook.
if project_root.name == 'notebooks':  # Nếu kernel đang ở notebooks/, lùi lên một cấp.
    project_root = project_root.parent  # Thư mục cha chính là gốc repository.
sys.path.insert(0, str(project_root))  # Thêm gốc dự án vào danh sách nơi Python tìm module.
print('Thư mục dự án:', project_root)  # In ra để người học xác nhận đúng workspace.


Thư mục dự án: c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project



Hai JSONL cần được tạo từ notebook EDA. Cần dùng môi trường Python đã cài các thư viện trong `requirements.txt`, đặc biệt là bản PyTorch có CUDA. Nếu không thấy GPU ở cell dưới, đừng chạy fine-tune vì CPU sẽ chậm hơn rất nhiều.


In [2]:
import torch  # PyTorch kiểm tra GPU và thực hiện phép tính của mô hình.
from src.train_qa import TRAIN_FILE, DEV_FILE  # Lấy đường dẫn dữ liệu mà script sẽ dùng.

print('Train tồn tại:', TRAIN_FILE.exists(), TRAIN_FILE)  # Xác nhận tệp train đã sẵn sàng.
print('Dev tồn tại:', DEV_FILE.exists(), DEV_FILE)  # Xác nhận tệp dev đã sẵn sàng.
print('Có CUDA:', torch.cuda.is_available())  # CUDA cho phép PyTorch dùng GPU NVIDIA.
if torch.cuda.is_available():  # Chỉ hỏi tên thiết bị khi đã phát hiện GPU.
    print('GPU:', torch.cuda.get_device_name(0))  # Kiểm tra notebook nhận đúng card
    print('Hỗ trợ BF16:', torch.cuda.is_bf16_supported())  # BF16 thường tiết kiệm bộ nhớ và ổn định.


c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train tồn tại: True C:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\data\processed\train.json
Dev tồn tại: True C:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\data\processed\dev.json
Có CUDA: True
GPU: NVIDIA GeForce RTX 5070 Ti
Hỗ trợ BF16: True


## 4. Chạy thí nghiệm mDeBERTa

Lượt này train lại seed 42, 123 và 1024 từ `microsoft/mdeberta-v3-base` trên train/dev đã làm sạch. Dùng cùng batch hiệu dụng 32 (4 × 8), ba epoch và hàm chấm như lượt XLM-R đã chốt.

Chạy cell dưới với `plan_only=True` để kiểm tra mã dữ liệu, revision, batch và đường dẫn. Đổi thành `False` để bắt đầu. Giữ cùng nhãn khi chạy tiếp lượt này.


In [6]:
from src.train_qa import run_experiment

# Ba seed độc lập dùng chung một cấu hình để so sánh mean và std.
seeds = [42, 123, 1024]
model_names = ["mdeberta"]
run_label = "mdeberta-batch32-clean-v1"
plan_only = False  # Xem cấu hình trước khi huấn luyện.

experiment_dir = run_experiment(
    model_names, seeds, plan_only=plan_only, run_label=run_label,
)


Phiên bản dữ liệu: uit-viquad2-05de3ab29680-ae347fbe9385-0bfbe5821468
Thư mục thí nghiệm: C:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\finetune\runs\run-a28388a1c7754133
Seed yêu cầu: [42, 123, 1024]
Nhãn lượt chạy: mdeberta-batch32-clean-v1
Batch hiệu dụng trên mỗi GPU: 4 x 8 = 32
mdeberta: revision a0484667b22365f84929a935b5e50a51f71f159d

Model: microsoft/mdeberta-v3-base | seed: 42
GPU: NVIDIA GeForce RTX 5070 Ti
Thư mục seed: C:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\finetune\runs\run-a28388a1c7754133\mdeberta\seed-42


Tạo nhãn start/end cho validation loss: 100%|██████████| 3814/3814 [00:01<00:00, 2809.25 examples/s]
c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(



--- Tóm tắt dữ liệu trước khi huấn luyện ---
Số mẫu train gốc: 8,000
Số mẫu dev gốc: 3,814
Số train features sau tiền xử lý: 11,881
Số dev features sau tiền xử lý: 5,493

Cấu hình huấn luyện:
Batch train trên mỗi GPU: 4
Số bước tích lũy gradient: 8
Batch hiệu dụng trên mỗi GPU: 4 x 8 = 32
Batch đánh giá trên mỗi GPU: 8
Số epoch: 3
Learning rate: 2e-05
Dùng BF16: True
Dùng FP16: False

Ví dụ nhãn của một train feature:
Feature index: 8
Start position: 105
End position: 130
Token được gán nhãn: ['▁ho', 'ạ', 't', '▁độ', 'ng', '▁v', 'ăn', '▁c', 'hương', ',', '▁', 'âm', '▁nh', 'ạc', ',', '▁ngh', 'ệ', '▁thu', 'ật', '▁', 'và', '▁', 'sân', '▁kh', 'ấ', 'u']
Đoạn giải mã từ nhãn: hoạt động văn chương, âm nhạc, nghệ thuật và sân khấu


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 38567.42it/s]
[transformers] DebertaV2ForQuestionAnswering LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
qa_outputs.bia

Epoch,Training Loss,Validation Loss,Exact Match,F1
1,5.953354,5.937500,17.435763,17.597566
2,5.741582,5.937500,17.435763,17.597566
3,0.000000,nan,30.440482,30.440482


Writing model shards: 100%|██████████| 1/1 [00:07<00:00,  7.11s/it]


... (8 outputs left out)

Epoch,Training Loss,Validation Loss,Exact Match,F1
1,5.357256,5.947214,14.997378,18.105707
2,4.204384,5.937500,17.435763,17.597566
3,3.649779,5.937500,17.435763,17.597566


Writing model shards: 100%|██████████| 1/1 [00:15<00:00, 15.24s/it]


Training Loss,Validation Loss,Epoch,Exact Match,F1
3.649779,5.947214,3,14.997378,18.105707


Kết quả seed 1024: EM=15.00, F1=18.11

Tổng hợp (F1/EM theo phần trăm):
mdeberta: seed tốt nhất=42, F1=30.44, EM=30.44; F1 trung bình=26.33 ± 5.81
Đã lưu tổng hợp tại: C:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\finetune\runs\run-a28388a1c7754133\summary-seeds-42-123-1024.json


## 5. Đọc kết quả trên dev

Tệp `summary-seeds-42-123-1024.json` lưu EM/F1 trung bình, độ lệch chuẩn và seed có F1 cao nhất. Đánh giá ở đây dùng dev có nhãn, cùng tập với lượt XLM-R.


In [7]:
# Tên summary gắn với đúng danh sách seed của lượt vừa chạy.
seed_suffix = "-".join(str(seed) for seed in sorted(set(seeds)))
summary_path = experiment_dir / model_names[0] / f"summary-seeds-{seed_suffix}.json"
if plan_only:
    print("Đang xem kế hoạch. Đặt plan_only=False khi muốn huấn luyện.")
elif summary_path.exists():
    print(summary_path.read_text(encoding="utf-8"))
else:
    print("Chưa có summary ở thư mục thí nghiệm:", summary_path)


{
  "experiment_id": "run-a28388a1c7754133",
  "dataset_version": "uit-viquad2-05de3ab29680-ae347fbe9385-0bfbe5821468",
  "requested_seeds": [
    42,
    123,
    1024
  ],
  "best_seed": 42,
  "best_f1": 30.440482433141057,
  "best_exact_match": 30.440482433141057,
  "mean_f1": 26.32889077894679,
  "std_f1": 5.814668680301557,
  "mean_exact_match": 25.292780982345743,
  "std_exact_match": 7.279949206762391
}


## 6. Lưu log và thống kê mô hình trả rỗng

Chạy cell này sau khi cả ba seed đã xong. Cell đọc dự đoán đã lưu, đếm chuỗi rỗng trên toàn dev và tách theo nhãn thật HasAns/NoAns. Nó cũng giữ lịch sử đánh giá của Trainer và cấu hình decoder. Tỷ lệ trả rỗng giúp phát hiện xu hướng của mô hình. Muốn biết tại sao một câu cụ thể trả rỗng, cần xem thêm mẫu và điểm logits của câu đó.


In [8]:
import json

from src.evaluation import MAX_ANSWER_LENGTH, N_BEST_TOKENS, NO_ANSWER_THRESHOLD
from src.utils import save_json

if plan_only:
    print("Chạy xong ba seed trước khi lưu thống kê.")
else:
    # Dùng đúng thư mục đã in ở bước train để chỉ đọc một phiên bản.
    model_dir = experiment_dir / "mdeberta"
    manifest = json.loads((experiment_dir / "run_manifest.json").read_text(encoding="utf-8"))
    diagnostics = {
        "experiment_id": manifest["experiment_id"],
        "dataset_version": manifest["data"]["dataset_version"],
        "split": "dev",
        "decoder": {
            "n_best_tokens": N_BEST_TOKENS,
            "max_answer_length": MAX_ANSWER_LENGTH,
            "no_answer_threshold": NO_ANSWER_THRESHOLD,
            "cls_aggregation": "min qua cac cua so",
            "no_answer_rule": "null_score - answer_score > threshold",
        },
        "prediction_rule": 'prediction_text == ""',
        "seeds": {},
    }

    for seed in sorted(set(seeds)):
        seed_dir = model_dir / f"seed-{seed}"
        metrics = json.loads((seed_dir / "metrics.json").read_text(encoding="utf-8"))
        predictions = json.loads((seed_dir / "predictions.json").read_text(encoding="utf-8"))
        if metrics["run_identity"]["experiment_fingerprint"] != manifest["fingerprint"]:
            raise ValueError(f"Seed {seed}: metrics thuộc lượt khác.")

        groups = {
            "HasAns": {"samples": 0, "empty_predictions": 0},
            "NoAns": {"samples": 0, "empty_predictions": 0},
        }
        for row in predictions:
            # Dùng nhãn thật để phân nhóm; chỉ chuỗi rỗng được đếm là trả rỗng.
            group = "NoAns" if row["is_impossible"] else "HasAns"
            groups[group]["samples"] += 1
            if row["prediction_text"] == "":
                groups[group]["empty_predictions"] += 1

        for stats in groups.values():
            stats["empty_rate_percent"] = (
                100 * stats["empty_predictions"] / stats["samples"] if stats["samples"] else None
            )

        # Checkpoint cuối lưu lịch sử tới epoch gần nhất; best_model có thể
        # được chọn ở epoch sớm hơn nên không dùng nó để đọc toàn bộ lịch sử.
        attempt_dir = seed_dir / metrics["training_attempt"]
        state_files = list(attempt_dir.glob("checkpoint-*/trainer_state.json"))
        eval_history = []
        if state_files:
            last_state = max(state_files, key=lambda file: int(file.parent.name.split("-")[-1]))
            trainer_state = json.loads(last_state.read_text(encoding="utf-8"))
            for entry in trainer_state["log_history"]:
                if "eval_f1" in entry:
                    eval_history.append(entry)
        else:
            print(f"Seed {seed}: không tìm thấy trainer_state.json để đọc lịch sử.")

        empty_total = groups["HasAns"]["empty_predictions"] + groups["NoAns"]["empty_predictions"]
        diagnostics["seeds"][str(seed)] = {
            "samples": len(predictions),
            "empty_predictions": empty_total,
            "empty_rate_percent": 100 * empty_total / len(predictions),
            "groups": groups,
            "eval_history": eval_history,
        }
        print(f"Seed {seed}: trả rỗng {empty_total}/{len(predictions)} câu dev.")

    diagnostics_path = model_dir / f"diagnostics-seeds-{seed_suffix}.json"
    save_json(diagnostics_path, diagnostics)
    print("Đã lưu thống kê mDeBERTa:", diagnostics_path)


Seed 42: trả rỗng 3814/3814 câu dev.
Seed 123: trả rỗng 3814/3814 câu dev.
Seed 1024: trả rỗng 1904/3814 câu dev.
Đã lưu thống kê mDeBERTa: C:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\finetune\runs\run-a28388a1c7754133\mdeberta\diagnostics-seeds-42-123-1024.json
